In [0]:
from pyspark.sql import functions as F

# Use the SAME storage account name that works in your Bronze/Silver notebook
account = "bikestores@oylamicdata.dfs.core.windows.net"
silver  = f"abfss://{account}/Silver"
gold    = f"abfss://{account}/Gold"

log_rows = []

def log(table, step, rows_in, rows_rejected, rows_out):
    log_rows.append((table, step, rows_in, rows_rejected, rows_out))
    print(f"[{table}] {step}: read={rows_in}, rejected={rows_rejected}, written={rows_out}")

In [0]:
orders     = spark.read.format("delta").load(f"{silver}/orders")
items      = spark.read.format("delta").load(f"{silver}/order_items")
products   = spark.read.format("delta").load(f"{silver}/products")
customers  = spark.read.format("delta").load(f"{silver}/customers")
stores     = spark.read.format("delta").load(f"{silver}/stores")
brands     = spark.read.format("delta").load(f"{silver}/brands")
categories = spark.read.format("delta").load(f"{silver}/categories")

n_items = items.count()

# order_status 3 = Rejected, so those orders are excluded from revenue
valid_orders = orders.filter(F.col("order_status") != 3)

sales = (items
    .join(valid_orders.select("order_id", "customer_id", "store_id", "order_date"), "order_id", "inner")
    .join(products.select("product_id", "product_name", "brand_id", "category_id"), "product_id", "inner")
    .join(brands, "brand_id", "left")
    .join(categories, "category_id", "left")
    .join(customers.select("customer_id", "first_name", "last_name", "state"), "customer_id", "left")
    .join(stores.select("store_id", "store_name", F.col("state").alias("store_state")), "store_id", "left")
    .withColumn("revenue", F.col("quantity") * F.col("list_price") * (1 - F.col("discount")))
)

n_sales = sales.count()
log("sales_fact", "join", n_items, n_items - n_sales, n_sales)

[sales_fact] join: read=4720, rejected=123, written=4597


In [0]:
valid_orders = orders.filter(F.col("order_status") != 3)

Daily, Monthly and Top Products

In [0]:
# 1. Daily sales
daily = (sales.groupBy("order_date")
         .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
              F.countDistinct("order_id").alias("total_orders"),
              F.sum("quantity").alias("units_sold")))
daily.write.mode("overwrite").format("delta").save(f"{gold}/daily_sales")
log("daily_sales", "aggregate", n_sales, 0, daily.count())

# 2. Monthly sales
monthly = (sales.withColumn("year", F.year("order_date"))
           .withColumn("month", F.month("order_date"))
           .groupBy("year", "month")
           .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
                F.countDistinct("order_id").alias("total_orders"),
                F.sum("quantity").alias("units_sold")))
monthly.write.mode("overwrite").format("delta").save(f"{gold}/monthly_sales")
log("monthly_sales", "aggregate", n_sales, 0, monthly.count())

# 3. Top products by revenue
top_products = (sales.groupBy("product_id", "product_name", "category_name", "brand_name")
                .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
                     F.sum("quantity").alias("units_sold")))
top_products.write.mode("overwrite").format("delta").save(f"{gold}/top_products")
log("top_products", "aggregate", n_sales, 0, top_products.count())

[daily_sales] aggregate: read=4597, rejected=0, written=711
[monthly_sales] aggregate: read=4597, rejected=0, written=29
[top_products] aggregate: read=4597, rejected=0, written=304


Customer Lifetime value and Store Revenue


In [0]:
# 4. Customer lifetime value
clv = (sales.groupBy("customer_id", "first_name", "last_name", "state")
       .agg(F.round(F.sum("revenue"), 2).alias("lifetime_value"),
            F.countDistinct("order_id").alias("total_orders")))
clv.write.mode("overwrite").format("delta").save(f"{gold}/customer_ltv")
log("customer_ltv", "aggregate", n_sales, 0, clv.count())

# 5. Revenue by store / region
by_store = (sales.groupBy("store_id", "store_name", "store_state")
            .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
                 F.countDistinct("order_id").alias("total_orders")))
by_store.write.mode("overwrite").format("delta").save(f"{gold}/revenue_by_store")
log("revenue_by_store", "aggregate", n_sales, 0, by_store.count())

[customer_ltv] aggregate: read=4597, rejected=0, written=1445
[revenue_by_store] aggregate: read=4597, rejected=0, written=3


In [0]:
log_df = spark.createDataFrame(
    log_rows, ["table", "step", "rows_read", "rows_rejected", "rows_written"]
)
log_df.write.mode("overwrite").format("delta").save(f"{gold}/_pipeline_log")
display(log_df)

table,step,rows_read,rows_rejected,rows_written
sales_fact,join,4720,123,4597
daily_sales,aggregate,4597,0,711
monthly_sales,aggregate,4597,0,29
top_products,aggregate,4597,0,304
customer_ltv,aggregate,4597,0,1445
revenue_by_store,aggregate,4597,0,3


Gold folders And rows check

In [0]:
# List every folder written to Gold
for f in dbutils.fs.ls(gold):
    print(f.name)

# Row count of each Gold table
for t in ["daily_sales", "monthly_sales", "top_products", "customer_ltv", "revenue_by_store"]:
    print(f"{t:18} rows = {spark.read.format('delta').load(f'{gold}/{t}').count()}")

# Quick business sanity check: revenue by store
display(spark.read.format("delta").load(f"{gold}/revenue_by_store").orderBy(F.desc("total_revenue")))

_pipeline_log/
customer_ltv/
daily_sales/
monthly_sales/
revenue_by_store/
top_products/
daily_sales        rows = 711
monthly_sales      rows = 29
top_products       rows = 304
customer_ltv       rows = 1445
revenue_by_store   rows = 3


store_id,store_name,store_state,total_revenue,total_orders
2,Baldwin Bikes,NY,5083946.42,1078
1,Santa Cruz Bikes,CA,1560539.33,337
3,Rowlett Bikes,TX,781523.30,155


In [0]:
root   = "abfss://bikestores@oylamicdata.dfs.core.windows.net"
silver = f"{root}/Silver"
spark.read.format("delta").load(f"{silver}/order_items").filter("order_id = 1616").show()

+--------+-------+----------+--------+----------+--------+
|order_id|item_id|product_id|quantity|list_price|discount|
+--------+-------+----------+--------+----------+--------+
|    1616|      2|         2|       2|      0.00|    0.10|
+--------+-------+----------+--------+----------+--------+



In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

for t in ["daily_sales", "monthly_sales", "top_products", "customer_ltv", "revenue_by_store"]:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS gold.{t}
        USING DELTA
        LOCATION '{gold}/{t}'
    """)

display(spark.sql("SHOW TABLES IN gold"))

database,tableName,isTemporary
gold,customer_ltv,false
gold,daily_sales,false
gold,monthly_sales,false
gold,revenue_by_store,false
gold,top_products,false


In [0]:
from pyspark.sql import functions as F

# Add a year column to the sales fact built earlier
sales = (sales
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("year", F.year("order_date")))
sales.cache()
n_sales = sales.count()

def write_gold(df, name, rows_in):
    out = df.count()
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .save(f"{gold}/{name}"))
    log(name, "aggregate", rows_in, 0, out)

# 1) revenue_by_store (with year)
revenue_by_store = (sales
    .groupBy("year", "store_id", "store_name", "store_state")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.round(F.sum("revenue"), 2).alias("total_revenue")))
write_gold(revenue_by_store, "revenue_by_store", n_sales)

# 2) top_products (with year)
top_products = (sales
    .groupBy("year", "product_id", "product_name", "brand_name", "category_name")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
         F.sum("quantity").alias("units_sold")))
write_gold(top_products, "top_products", n_sales)

# 3) customer_ltv (with year)
customer_ltv = (sales
    .groupBy("year", "customer_id", "first_name", "last_name", "state")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.round(F.sum("revenue"), 2).alias("lifetime_value")))
write_gold(customer_ltv, "customer_ltv", n_sales)

# Check: all three totals must match
for name, df, col in [("store", revenue_by_store, "total_revenue"),
                      ("products", top_products, "total_revenue"),
                      ("customers", customer_ltv, "lifetime_value")]:
    print(f"{name}: {df.agg(F.sum(col)).first()[0]:,.2f}")

[revenue_by_store] aggregate: read=4597, rejected=0, written=9
[top_products] aggregate: read=4597, rejected=0, written=432
[customer_ltv] aggregate: read=4597, rejected=0, written=1557
store: 7,426,009.06
products: 7,426,008.87
customers: 7,426,008.05


In [0]:
%sql
-- Q2: Top 10 products by revenue (all years combined)
SELECT product_name, brand_name, category_name,
       ROUND(SUM(total_revenue), 2) AS revenue,
       SUM(units_sold) AS units
FROM gold.top_products
GROUP BY product_id, product_name, brand_name, category_name
ORDER BY revenue DESC
LIMIT 10

product_name,brand_name,category_name,revenue,units
Trek Slash 8 27.5 - 2016,Trek,Mountain Bikes,551758.62,153
Trek Conduit+ - 2016,Trek,Electric Bikes,386398.71,144
Trek Fuel EX 8 29 - 2016,Trek,Mountain Bikes,357568.77,139
Surly Straggler 650b - 2016,Surly,Cyclocross Bicycles,223571.67,149
Trek Remedy 29 Carbon Frameset - 2016,Trek,Mountain Bikes,199996.89,123
Surly Straggler - 2016,Surly,Cyclocross Bicycles,196908.88,142
Trek Domane SLR 6 Disc - 2017,Trek,Road Bikes,186229.67,38
Trek Powerfly 8 FS Plus - 2017,Trek,Electric Bikes,179249.64,39
Trek Madone 9.2 - 2017,Trek,Road Bikes,175899.64,39
Trek Silque SLR 8 Women's - 2017,Trek,Road Bikes,174524.73,29


In [0]:
%sql
-- Q3: Monthly revenue and month-over-month growth
SELECT make_date(year, month, 1) AS month_start,
       total_revenue AS revenue,
       total_orders,
       ROUND(total_revenue - LAG(total_revenue) OVER (ORDER BY year, month), 2) AS mom_change,
       ROUND(100 * (total_revenue / LAG(total_revenue) OVER (ORDER BY year, month) - 1), 1) AS mom_pct
FROM gold.monthly_sales
ORDER BY month_start

month_start,revenue,total_orders,mom_change,mom_pct
2016-01-01,215146.42,50,null,null
2016-02-01,148506.36,47,-66640.06,-31.0
2016-03-01,180432.23,55,31925.87,21.5
2016-04-01,164399.08,41,-16033.15,-8.9
2016-05-01,197179.09,49,32780.01,19.9
2016-06-01,205463.98,44,8284.89,4.2
2016-07-01,199556.81,50,-5907.17,-2.9
2016-08-01,212827.11,59,13270.30,6.6
2016-09-01,271356.68,66,58529.57,27.5
2016-10-01,212078.08,64,-59278.60,-21.8


In [0]:
%sql
-- Q4: Top 20 customers by lifetime value (all years combined)
SELECT customer_id,
       CONCAT(first_name, ' ', last_name) AS customer,
       state,
       SUM(total_orders) AS orders,
       ROUND(SUM(lifetime_value), 2) AS lifetime_value
FROM gold.customer_ltv
GROUP BY customer_id, first_name, last_name, state
ORDER BY lifetime_value DESC
LIMIT 20

customer_id,customer,state,orders,lifetime_value
94,Sharyn Hopkins,NY,2,34807.93
75,Abby Gamble,NY,2,32803.01
73,Melanie Hayes,NY,2,31913.69
93,Corrina Sawyer,NY,2,25612.71
1224,Abram Copeland,TX,1,24607.03
6,Lyndsey Bean,NY,2,23711.11
85,Teofila Fischer,NY,2,23195.08
1,Debra Burks,NY,3,22630.22
35,Guillermina Noble,NY,2,22061.07
38,Zelma Browning,NY,2,21535.66


In [0]:
%sql
-- Q5: Customer value by state
SELECT state,
       COUNT(DISTINCT customer_id) AS customers,
       ROUND(SUM(lifetime_value), 2) AS total_ltv,
       ROUND(SUM(lifetime_value) / COUNT(DISTINCT customer_id), 2) AS ltv_per_customer
FROM gold.customer_ltv
GROUP BY state
ORDER BY total_ltv DESC

state,customers,total_ltv,ltv_per_customer
NY,1019,5083945.78,4989.15
CA,284,1560539.05,5494.86
TX,142,781523.22,5503.68


In [0]:
%sql
-- Q6: Revenue by category and year
SELECT year, category_name, ROUND(SUM(total_revenue), 2) AS revenue
FROM gold.top_products
GROUP BY year, category_name
ORDER BY year, revenue DESC

year,category_name,revenue
2016,Mountain Bikes,1160084.50
2016,Cruisers Bicycles,369482.94
2016,Cyclocross Bicycles,327716.15
2016,Electric Bikes,277319.08
2016,Comfort Bicycles,151412.72
2016,Children Bicycles,86665.72
2017,Mountain Bikes,1096316.06
2017,Road Bikes,1004578.18
2017,Cruisers Bicycles,371007.89
2017,Cyclocross Bicycles,285443.86


In [0]:
%sql
SELECT MIN(order_date) AS first_day, MAX(order_date) AS last_day, COUNT(*) AS days
FROM gold.daily_sales

first_day,last_day,days
2016-01-01,2018-06-01,711


In [0]:
%sql
SELECT order_date, total_revenue, total_orders, units_sold
FROM gold.daily_sales
ORDER BY order_date DESC
LIMIT 40

order_date,total_revenue,total_orders,units_sold
2018-06-01,0.00,1,2
2018-04-30,3660.62,2,9
2018-04-29,24320.86,4,17
2018-04-28,7640.54,4,8
2018-04-27,55129.07,5,34
2018-04-26,39376.13,5,30
2018-04-25,4988.43,3,8
2018-04-24,6134.12,2,9
2018-04-23,40227.68,6,36
2018-04-22,13678.70,3,12


In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_id) AS customers FROM gold.customer_ltv

rows,customers
1557,1445
